# 🤖 AI-Powered Company Brochure Generator

This project uses **OpenAI GPT models** to automatically generate company brochures by:
1. Scraping a company's website and discovering relevant links
2. Using an LLM to intelligently filter which links are worth including
3. Aggregating all content and generating a polished markdown brochure
4. Streaming the final output for a smooth user experience

> **Skills demonstrated:** Prompt engineering, structured JSON outputs, web scraping, streaming LLM responses, multi-step agentic pipelines


## 1. Setup & Imports

In [ ]:
import os
import json

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

from scraper import fetch_website_links, fetch_website_contents

In [ ]:
# Load environment variables from .env file
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("✅ API key loaded successfully")
else:
    print("⚠️  API key not found or invalid. Please check your .env file.")

# Model configuration
LINK_FILTER_MODEL = 'gpt-5-nano'   # Fast, cheap model for link filtering
BROCHURE_MODEL = 'gpt-4.1-mini'    # Higher-quality model for content generation

openai = OpenAI()

## 2. Step 1 — Filter Relevant Links with an LLM

Rather than crawling every link on a page, we ask a small LLM to intelligently decide which links are worth including in a brochure (e.g. About, Careers, Products). This is a great use case for LLMs because it requires nuanced understanding that would be very difficult to hard-code.

We use **one-shot prompting** — providing an example response directly in the system prompt — so the model understands the expected JSON format.

In [ ]:
LINK_SYSTEM_PROMPT = """
You are provided with a list of links found on a webpage.
Decide which links are most relevant to include in a company brochure,
such as links to an About page, Company page, or Careers/Jobs page.

Respond only in JSON using this format:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}

Always return full https:// URLs, converting any relative links.
"""


def get_links_user_prompt(url: str) -> str:
    """Build a user prompt listing all links scraped from the given URL."""
    links = fetch_website_links(url)
    prompt = (
        f"Here is the list of links found on the website: {url}\n\n"
        "Please decide which are relevant for a company brochure. "
        "Return full https URLs in JSON format. "
        "Exclude: Terms of Service, Privacy Policy, and email links.\n\n"
        "Links:\n"
    )
    prompt += "\n".join(links)
    return prompt


def select_relevant_links(url: str) -> dict:
    """Call the LLM to filter and return only brochure-relevant links."""
    print(f"🔍 Selecting relevant links for {url} using {LINK_FILTER_MODEL}...")
    response = openai.chat.completions.create(
        model=LINK_FILTER_MODEL,
        messages=[
            {"role": "system", "content": LINK_SYSTEM_PROMPT},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = json.loads(response.choices[0].message.content)
    print(f"✅ Found {len(result['links'])} relevant links")
    return result

In [ ]:
# Test link selection
select_relevant_links("https://huggingface.co")

## 3. Step 2 — Aggregate Page Content

We fetch the main landing page and each relevant linked page, then combine their contents into a single context string to pass to the brochure-writing LLM.

In [ ]:
def fetch_page_and_all_relevant_links(url: str) -> str:
    """Scrape the landing page and all relevant linked pages into a single string."""
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### {link['type'].title()}\n"
        result += fetch_website_contents(link["url"])

    return result

## 4. Step 3 — Generate the Brochure

With all the page content assembled, we prompt a higher-quality model to write a polished company brochure in markdown.

The `stream_brochure` function streams tokens back in real time, producing the familiar typewriter effect in the notebook.

In [ ]:
BROCHURE_SYSTEM_PROMPT = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors, and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers, and careers/jobs if the information is available.
"""

# Uncomment below for a more entertaining, witty tone:
# BROCHURE_SYSTEM_PROMPT = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, witty brochure about the company for prospective customers, investors, and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers, and careers/jobs if the information is available.
# """


def get_brochure_user_prompt(company_name: str, url: str) -> str:
    """Build the user prompt containing all scraped page content."""
    prompt = (
        f"You are looking at a company called: {company_name}\n"
        "Here are the contents of its landing page and other relevant pages. "
        "Use this information to build a short brochure in markdown.\n\n"
    )
    prompt += fetch_page_and_all_relevant_links(url)
    return prompt[:5_000]  # Truncate to stay within token limits


def stream_brochure(company_name: str, url: str) -> None:
    """Generate and stream a company brochure directly into the notebook."""
    stream = openai.chat.completions.create(
        model=BROCHURE_MODEL,
        messages=[
            {"role": "system", "content": BROCHURE_SYSTEM_PROMPT},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

## 5. Run It!

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try it with any company!
stream_brochure("Anthropic", "https://anthropic.com")